## Análise de Desempenho de Atendimento (Camada Gold)
Este notebook tem como objetivo principal processar os dados de chamados da camada Silver para criar tabelas e visões na camada Gold, focadas na análise de desempenho de atendentes e níveis de atendimento. As métricas chave incluem quantidade de chamados, taxa de resolução, CSAT médio (nota de atendimento) e tempo médio de atendimento.

### 1. Preparação e Carregamento de Dados

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{gold_db_name}")

df_geral = spark.read.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral")


In [0]:
df_geral.printSchema()

### 2. Criação da Tabela Fato de Atendimentos por Atendente (ft_atendimentos)
Objetivo: Agrupar os dados por id_atendente para calcular métricas de desempenho agregadas e persistir como a tabela fato principal na camada Gold.

In [0]:
# Criar a tabela fato de atendimentos
ft_atendimentos = df_geral.groupBy("id_atendente").agg(
 
    F.count("id_chamado").alias("qtd_chamados"), #qtd chamados por atendente

    F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)).alias("qtd_chamados_resolvidos"),
    
    F.avg("tempo_atendimento_segundos").alias("tempo_medio_atendimento"), #tempo médio de atendimento
    
    # percentual de casos resolvidos
    (F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)) / 
     F.count("id_chamado") * 100).alias("taxa_resolucao"),
    
    # nota_atendimento média
    F.avg("nota_atendimento").alias("csat_medio"),
    
    # custo médio por chamado
    F.avg("valor_custo").alias("custo_medio_por_chamado"),
    
    # ADICIONAR nivel_atendimento (pegar o primeiro valor, assumindo que é fixo por atendente)
    F.first("nivel_atendimento").alias("nivel_atendimento")
)

# 2 casas decimais apenas
ft_atendimentos = ft_atendimentos.select(
    "id_atendente",
    "qtd_chamados",
    "qtd_chamados_resolvidos",
    "nivel_atendimento",  # Agora existe na agregação
    F.round("tempo_medio_atendimento", 2).alias("tempo_medio_atendimento"),
    F.round("taxa_resolucao", 2).alias("taxa_resolucao"),
    F.round("csat_medio", 2).alias("csat_medio"),
    F.round("custo_medio_por_chamado", 2).alias("custo_medio_por_chamado")
)

# adicionar coluna de data de processamento
ft_atendimentos = ft_atendimentos.withColumn(
    "data_processamento",
    F.current_timestamp()
)

# Ordenar por quantidade de chamados
ft_atendimentos = ft_atendimentos.orderBy(F.desc("qtd_chamados"))

# Visualizar o resultado
display(ft_atendimentos)

print("\nSchema da FT_ATENDIMENTOS:")
ft_atendimentos.printSchema()

# salvar a tabela fato
ft_atendimentos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{gold_db_name}.ft_atendimentos")

### 3. Criação da Visão de Top Performers
Objetivo: Identificar os atendentes com o melhor desempenho geral, utilizando um Score Composto de Desempenho.

In [0]:
vw_top_performers = ft_atendimentos.filter(
    F.col("qtd_chamados") >= 100
).select(
    "id_atendente",
    "qtd_chamados",
    "nivel_atendimento",
    F.round("csat_medio", 2).alias("csat_medio"),
    F.round("taxa_resolucao", 2).alias("taxa_resolucao"),
    F.round(F.col("tempo_medio_atendimento") / 60, 2).alias("tempo_medio_minutos"),
    # Score composto de desempenho (peso: CSAT 40%, Taxa Resolução 40%, Velocidade 20%)
    F.round(
        (F.col("csat_medio") / 5 * 0.4) +  # Normaliza nota de 0-5 para 0-1
        (F.col("taxa_resolucao") / 100 * 0.4) +  # Normaliza % para 0-1
        (F.greatest(F.lit(0), 1 - (F.col("tempo_medio_atendimento") / 600)) * 0.2),  # Penaliza > 10min
        3
    ).alias("score_desempenho")
).orderBy(F.desc("score_desempenho"))

In [0]:
vw_top_performers.createOrReplaceTempView("vw_top_performers")

resultado = spark.sql("""
    SELECT 
        id_atendente,
        qtd_chamados,
        nivel_atendimento,
        csat_medio,
        score_desempenho
    FROM vw_top_performers
    WHERE qtd_chamados > 100
    ORDER BY score_desempenho DESC
    LIMIT 10
""")

display(resultado)

### 4. Criação da Visão de Atendentes de Risco
Objetivo: Identificar atendentes que estão abaixo dos critérios de performance e classificá-los por tipo de risco e prioridade de ação.

In [0]:
vw_atendentes_risco = ft_atendimentos.filter(
    (F.col("csat_medio") < 3) |  
    (F.col("taxa_resolucao") < 70) |
    (F.col("tempo_medio_atendimento") > 120)
).select(
    "id_atendente",
    "qtd_chamados",
    "nivel_atendimento",
    F.round("csat_medio", 2).alias("csat_medio"),
    F.round("taxa_resolucao", 2).alias("taxa_resolucao"),
    F.round(F.col("tempo_medio_atendimento"), 2).alias("tempo_medio_atendimento"),
    F.when(
        (F.col("csat_medio") < 3) &
        (F.col("taxa_resolucao") < 70) &
        (F.col("tempo_medio_atendimento") > 120),
        "MULTIPLOS_RISCOS"
    ).when(
        F.col("csat_medio") < 3, "CSAT_BAIXO"
    ).when(
        F.col("taxa_resolucao") < 70, "BAIXA_RESOLUCAO"
    ).when(
        F.col("tempo_medio_atendimento") > 120, "LENTO"
    ).alias("tipo_risco")

).orderBy(F.asc("csat_medio"))


In [0]:
vw_atendentes_risco.createOrReplaceTempView("vw_atendentes_risco")

resultado = spark.sql("""
    SELECT 
        id_atendente,
        qtd_chamados,
        nivel_atendimento,
        csat_medio,
        taxa_resolucao,
        tempo_medio_atendimento,
        tipo_risco,
        CASE 
            WHEN tipo_risco = 'MULTIPLOS_RISCOS' THEN 'ALTA'
            WHEN csat_medio < 2.95 AND taxa_resolucao < 65 THEN 'ALTA'
            WHEN csat_medio < 3.0 OR taxa_resolucao < 70 THEN 'MÉDIA'
            ELSE 'BAIXA'
        END AS prioridade_acao,
        -- Gap para meta
        ROUND(3 - csat_medio, 2) AS gap_csat,
        ROUND(70.0 - taxa_resolucao, 2) AS gap_taxa_resolucao
    FROM vw_atendentes_risco
    ORDER BY 
        CASE tipo_risco 
            WHEN 'MUlTIPLOS_RISCOS' THEN 1
            WHEN 'CSAT_BAIXO' THEN 2
            WHEN 'BAIXA_RESOLUCAO' THEN 3
            WHEN 'LENTO' THEN 4
        END,
        csat_medio ASC,
        qtd_chamados DESC
    LIMIT 20
""")

display(resultado)

### 5. Criação da Visão de Desempenho por Nível
Objetivo: Analisar o desempenho e o custo do atendimento, agregando as métricas pela coluna nivel_atendimento.

In [0]:
vw_desempenho_por_nivel = ft_atendimentos.groupBy("nivel_atendimento").agg(
    # Quantidade de atendentes por nível
    F.count("id_atendente").alias("qtd_atendentes"),
    
    # Total de chamados por nível
    F.sum("qtd_chamados").alias("total_chamados"),

    F.sum("qtd_chamados_resolvidos").alias("total_chamados_resolvidos"),
    
    # Média de chamados por atendente
    F.round(F.avg("qtd_chamados"), 2).alias("media_chamados_por_atendente"),
    
    # Média da taxa de resolução por nível
    F.round(F.avg("taxa_resolucao"), 2).alias("taxa_resolucao_media"),
    
    # Média do CSAT por nível
    F.round(F.avg("csat_medio"), 2).alias("csat_medio"),
    
    # Média do tempo de atendimento por nível (em minutos)
    F.round(F.avg("tempo_medio_atendimento") / 60, 2).alias("tempo_medio_minutos"),
    
    # Custo médio por nível
    F.round(F.avg("custo_medio_por_chamado"), 2).alias("custo_medio")
    
).orderBy("nivel_atendimento")

In [0]:
vw_desempenho_por_nivel.createOrReplaceTempView("vw_desempenho_por_nivel")

resultado = spark.sql("""
    SELECT 
        nivel_atendimento,
        qtd_atendentes,
        total_chamados,
        total_chamados_resolvidos,
        taxa_resolucao_media,
        csat_medio,
        tempo_medio_minutos,

        ROUND(tempo_medio_minutos * total_chamados,2 ) AS tempo_total_gasto_minutos,
        custo_medio,
        
        -- Custo Total
        ROUND(custo_medio * total_chamados, 2) AS custo_total,
        
        -- Outras métricas derivadas úteis
        ROUND(custo_medio * total_chamados_resolvidos, 2) AS custo_total_resolvidos,
        ROUND(custo_medio * (total_chamados - total_chamados_resolvidos), 2) AS custo_total_nao_resolvidos,
        
        -- Custo por chamado resolvido
        ROUND((custo_medio * total_chamados) / total_chamados_resolvidos, 2) AS custo_por_resolvido
        
    FROM vw_desempenho_por_nivel
    ORDER BY nivel_atendimento
""")

display(resultado)

### 6. Persistência Final das Visões
Objetivo: Salvar as visões e a tabela fato criadas na camada Gold como tabelas Delta, garantindo que os dados analíticos estejam disponíveis para consumo

In [0]:
vw_top_performers.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{gold_db_name}.vw_top_performers")

vw_atendentes_risco.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{gold_db_name}.vw_atendentes_risco")

vw_desempenho_por_nivel.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{gold_db_name}.vw_desempenho_por_nivel")